**Install VDK and the SQLite plugin**

In [1]:
!pip install vdk-core vdk-sqlite requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for vdk-core: filename=vdk_core-0.3.1466514302-py2.py3-none-any.whl size=163460 sha256=4c11c764993c7f81827f3ccc2de2f74f50cd4e6ab68b291f881a99b2060eedba
  Stored in directory: /root/.cache/pip/wheels/ee/3f/c8/010983f3969c74948165dbfb7ab7621cdf4ff619cc0aca83c2
  Created wheel for vdk-sqlite: filename=vdk_sqlite-0.1.1431637373-py3-none-any.whl size=7065 sha256=c33784b1cbd3b9ba9e8dc4df163d952762180c7a24c8e9fe6934514da0257c90
  Stored in directory: /root/.cache/pip/wheels/1e/fa/b3/11d78000303a2c009ad8ca4679a5b8b210a9074ae78956f045
Successfully built vdk-core vdk-sqlite


**Set Environment Variables for SQLite Ingestion**

In [2]:
import os
os.environ["VDK_DB_DEFAULT_TYPE"] = "SQLITE"
os.environ["VDK_INGEST_METHOD_DEFAULT"] = "SQLITE"

In [3]:
%%writefile 10_delete_table.sql
DROP TABLE IF EXISTS rest_target_table;

Writing 10_delete_table.sql


In [4]:
%%writefile 20_create_table.sql
CREATE TABLE rest_target_table (userId, id, title, completed);

Writing 20_create_table.sql


In [5]:
%%writefile 30_rest_ingest.py
import requests

def run(job_input):
    # Fetching from the API
    response = requests.get("https://jsonplaceholder.typicode.com/todos/1")
    response.raise_for_status()
    payload = response.json()

    # Sending data to the target database
    job_input.send_object_for_ingestion(
        payload=payload,
        destination_table="rest_target_table"
    )

Writing 30_rest_ingest.py


In [6]:
!vdk run .


Versatile Data Kit (VDK)
Version: 0.3.1466514302
Build details: RELEASE_VERSION=0.3.1466514302, BUILD_DATE=Tue Sep 24 08:50:56 UTC 2024, BUILD_MACHINE_INFO=Linux runner--azerasq-project-28359933-concurrent-0 5.15.154+ #1 SMP Sat May 4 12:14:42 UTC 2024 x86_64 GNU/Linux, GITLAB_CI_JOB_ID=7902585266, GIT_COMMIT_SHA=880df089916daa16d3fe5fbe789b81dd6099911f, GIT_BRANCH=main
Python version: 3.12.12 64bit (/usr/bin/python3)

Installed plugins:
vdk-sqlite (from package vdk-sqlite, version 0.1.1431637373)
--------------------------------------------------------------------------------
Run job with directory /content
Missing config.ini file.
2026-01-10 01:50:42,039 [VDK] content [INFO ] vdk.plugin.sqlite.sqlite_conne sqlite_connection.py:29   new_connection  [id:317624ed-7ba7-4632-af5c-9d04c87281d1-1768009841-51ceb]- Creating new connection against local file database located at: /tmp/vdk-sqlite.db
2026-01-10 01:50:42,039 [VDK] content [INFO ] vdk.plugin.sqlite.sqlite_conne sqlite_connection.p

In [7]:
!vdk sqlite-query -q 'SELECT * FROM rest_target_table'


Creating new connection against local file database located at: /tmp/vdk-sqlite.db
  userId    id  title                 completed
--------  ----  ------------------  -----------
       1     1  delectus aut autem            0
